# 🏠 HouseCrafter: Lifting 2D Floorplans to 3D Indoor Scenes
### Interactive Gradio Web UI with 3D PLY Viewer & Google Drive Auto-Sync

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sourman-dev/houseCrafter/blob/feat/gradio-colab-ui/notebooks/HouseCrafter_Gradio_Colab.ipynb)
[![Project Page](https://img.shields.io/badge/Project-Page-blue)](https://neu-vi.github.io/houseCrafter/)
[![arXiv](https://img.shields.io/badge/arXiv-2406.20077-b31b1b.svg)](https://arxiv.org/abs/2406.20077)

---

## 📖 Overview
This notebook sets up and launches the **HouseCrafter** Gradio web application on Google Colab:
1. **Input**: Upload a 2D floorplan image (PNG/JPG) or select from preset 3D-Front dataset samples.
2. **Process**: Multi-view RGB-D diffusion sampling + TSDF volume fusion + mesh denoising.
3. **Output**: Interactive 3D `.ply` model in your browser with direct file download.
4. **Google Drive Sync**: Automatically synchronizes all outputs to `Gradio/houseCrafter/output` on your Google Drive.

## ⚡ Step 1: Check GPU & Hardware Diagnostics

In [ ]:
!nvidia-smi
import torch
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"[*] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 💾 Step 2: Mount Google Drive & Setup Output Folder
Mount your Google Drive to enable automatic persistence of all generated 3D `.ply` models and multi-view RGB-D images.

In [ ]:
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Auto-create target directory: Gradio/houseCrafter/output
gdrive_output_dir = '/content/drive/MyDrive/Gradio/houseCrafter/output'
os.makedirs(gdrive_output_dir, exist_ok=True)
print(f"[OK] Google Drive output folder ready: {gdrive_output_dir}")
## 📥 Step 3: Clone HouseCrafter Repository from GitHub
Clone the repository and checkout the `feat/gradio-colab-ui` branch.

In [ ]:
%cd /content
!rm -rf /content/houseCrafter
!git clone https://github.com/sourman-dev/houseCrafter.git || git clone https://github.com/neu-vi/houseCrafter.git
%cd /content/houseCrafter
!git checkout feat/gradio-colab-ui || git pull

## 📦 Step 4: Install Dependencies & Setup Environment

In [ ]:
%cd /content/houseCrafter
!bash scripts/colab_setup.sh

## 🎯 Step 5: Download Model Checkpoints & Sample Data
Download the pre-trained diffusion weights (`ckpts/`) and sample floorplans (`dataRelease/`).
If you already saved checkpoints in your Drive (`MyDrive/houseCrafter_ckpts`), this cell will copy them automatically to save bandwidth.

In [ ]:
%cd /content/houseCrafter
import os

# Check if weights exist in Google Drive cache
gdrive_ckpt_cache = '/content/drive/MyDrive/houseCrafter_ckpts'
if os.path.exists(gdrive_ckpt_cache):
    print("[*] Copying checkpoints from Google Drive cache...")
    !cp -r /content/drive/MyDrive/houseCrafter_ckpts/* ckpts/
else:
    print("[*] Downloading pre-trained checkpoints via gdown...")
    # Pre-trained checkpoint download (Google Drive link from README)
    !gdown --folder https://drive.google.com/drive/folders/1OY_V9nV5kOfGLa6oSlZMzVp0vRst2g3Y -O ckpts/ || echo "[Notice] Download manually if quota throttled."

# Sample data download
if not os.path.exists('dataRelease/layout_samples'):
    print("[*] Downloading sample data...")
    !gdown --folder https://drive.google.com/drive/folders/18p5m_RN5O9zDNe80ertQJPjEDTqAqTM- -O dataRelease/ || echo "[Notice] Sample data download completed."

# Run verification diagnostics
!python scripts/verify_env.py

## 🚀 Step 6: Launch Gradio Web Application
Run the command below. Gradio will generate a **public URL** (`https://xxxx.gradio.live`) that you can open in any browser.

In [ ]:
%cd /content/houseCrafter

# Launch Gradio with public share link & direct sync to Google Drive
# Add --mock if you want to test UI without downloading 24GB weights
!python app.py --share --gdrive_dir "/content/drive/MyDrive/Gradio/houseCrafter/output"

## 📂 Step 7: Inspect Synced Outputs in Google Drive

In [ ]:
import os
gdrive_out = '/content/drive/MyDrive/Gradio/houseCrafter/output'
if os.path.exists(gdrive_out):
    scenes = sorted(os.listdir(gdrive_out))
    print(f"[*] Total scenes saved in Google Drive: {len(scenes)}")
    for s in scenes[-5:]:
        print(f"  📁 {s}")
else:
    print("No outputs found yet.")